In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

### 학습, 검증, 테스트 데이터의 역할

In [2]:
from sklearn.datasets import load_iris # iris 데이터 로드

iris = load_iris()
X = iris.data
y = iris.target

display(X)
display(y)

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [3]:
# 1차 분할 : 전체 데이터를 [학습 + 검증] vs [테스트]로 분할 (8 : 2)
X_train_val, X_test, y_train_val, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [4]:
# 2차 분할 : [학습 + 검증] 데이터를 [학습] vs [검증]으로 분리 (8 : 2)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42)

In [5]:
# 학습용
print(f'학습 데이터 : {X_train.shape}')
# 학습된 모델의 성능을 미세조정하기 위해
print(f'검증 데이터 : {X_val.shape}')
# 최종 검증용
print(f'테스트 데이터 : {X_test.shape}')

학습 데이터 : (96, 4)
검증 데이터 : (24, 4)
테스트 데이터 : (30, 4)


### K Fole 교차 검증
- 데이터가 부족할 때 사용하면 좋은 방법이다.
- 데이터를 K 개의 Fold(꾸러미)로 나눠 학습용과 평가용 데이터를 변경하면서 K 번째 평가를 하는 방법

In [6]:
# KFold 객체를 생성한다.
# n_splits : 데이터 꾸러미의 개수, 이 개수 만큼 학습과 평가를 수행한다.
kfold = KFold(n_splits=5, shuffle=True)
list(kfold.split(X,y))

[(array([  2,   3,   4,   5,   7,   9,  10,  11,  12,  14,  15,  16,  17,
          18,  19,  20,  21,  22,  24,  25,  26,  27,  28,  29,  30,  31,
          32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,
          45,  46,  49,  51,  52,  53,  54,  55,  56,  58,  61,  62,  63,
          64,  65,  66,  69,  70,  71,  72,  74,  76,  77,  78,  79,  80,
          82,  84,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,
          99, 100, 102, 103, 104, 105, 106, 108, 110, 113, 114, 115, 116,
         117, 120, 121, 122, 123, 124, 125, 126, 128, 129, 130, 131, 132,
         133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146,
         147, 148, 149]),
  array([  0,   1,   6,   8,  13,  23,  47,  48,  50,  57,  59,  60,  67,
          68,  73,  75,  81,  83,  85,  86,  87, 101, 107, 109, 111, 112,
         118, 119, 127, 135])),
 (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
          13,  14,  15,  16,  17,  18,  19,  20,  21, 

In [8]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(random_state=42)

#kfold 를 생성한다.
kfold = KFold(n_splits=5)
scores = cross_val_score(dt_clf, X, y, scoring="accuracy", cv=kfold)

print(f'5회의 평가 결과 : {scores}')
print(f'5회의 평가 결과 평균 : {np.mean(scores)}')

5회의 평가 결과 : [1.         1.         0.83333333 0.93333333 0.8       ]
5회의 평가 결과 평균 : 0.9133333333333333


### Stratified K-Fold
- K-Fold 는 회귀의 문제를 해결하고자 할 때 사용
- Stratified K-Fold는 분류의 문제를 해결하고자 할 때 사용
- 학습용과 평가용 데이터의 결과 데이터 비율이 비슷하게 될 수 있도록 구성을해준다

In [ ]:
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['label'] = iris.target
iris_df

In [ ]:
# 1. 일반 K-Fold (Shuffle=False)
kfold = KFold(n_splits=3)
list(kfold.split(iris_df))

for train_idx, test_idx in kfold.split(iris_df) :
    label_train = iris_df['label'].iloc[train_idx]
    label_test = iris_df['label'].iloc[test_idx]
    print(f'학습용 데이터의 결과데이터 구성')
    print(label_train.value_counts())
    print(f'평가용 데이터의 결과데이터 구성')
    print(label_test.value_counts())
    print('='*30)

In [ ]:
# 일반 K-Fold (Shuffle=True)
kfold = KFold(n_splits=3, shuffle=True)

for train_idx, test_idx in kfold.split(iris_df) :
    label_train = iris_df['label'].iloc[train_idx]
    label_test = iris_df['label'].iloc[test_idx]
    print(f'학습용 데이터의 결과데이터 구성')
    print(label_train.value_counts())
    print(f'평가용 데이터의 결과데이터 구성')
    print(label_test.value_counts())
    print('='*30)

In [ ]:
# Stritified K-Fold
skf = StratifiedKFold(n_splits=3)

for train_idx, test_idx in skf.split(iris_df, iris_df['label']) :
    label_train = iris_df['label'].iloc[train_idx]
    label_test = iris_df['label'].iloc[test_idx]
    print(f'학습용 데이터의 결과데이터 구성')
    print(label_train.value_counts())
    print(f'평가용 데이터의 결과데이터 구성')
    print(label_test.value_counts())
    print('='*30)


In [10]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(random_state=42)

#Stritified K-Fold 를 생성한다.
skf = StratifiedKFold(n_splits=5)
scores = cross_val_score(dt_clf, X, y, scoring="accuracy", cv=skf)

print(f'5회의 평가 결과 : {scores}')
print(f'5회의 평가 결과 평균 : {np.mean(scores)}')

5회의 평가 결과 : [0.96666667 0.96666667 0.9        0.93333333 1.        ]
5회의 평가 결과 평균 : 0.9533333333333334
